# 08 - From DFIT to Fracture Treatment Design

This notebook closes the loop. A DFIT is run to calibrate four inputs
that a hydraulic fracture simulator needs:

1. **Closure stress** (minimum in-situ stress) - sets the minimum pressure
   to hold the fracture open.
2. **Fluid efficiency** (eta) - determines how much fluid you need to inject
   to create a fracture of a given size.
3. **Leakoff coefficient** (C_L) - quantifies how fast fluid leaks from the
   fracture into the formation.
4. **Net pressure** - constrains the fracture width model.

Without the DFIT, engineers guess these four numbers. With it, they are
measured. The improvement in fracture geometry prediction can be the
difference between a 200 ft and a 600 ft effective half-length.

This module uses PKN (Perkins-Kern-Nordgren) geometry, which is the
standard for moderate-to-high-net-pressure confined-height fractures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dfit
from dfit.fracdesign import fracture_design_from_dfit, design_summary

## 1. Run the DFIT and interpret

In [ ]:
d = dfit.generate_dfit(
    regime="pressure_dependent",
    t_pump_min=6.0,
    isip_psi=8400,
    closure_pressure_psi=7100,
    closure_G=7.0,
    reservoir_pressure_psi=5800,
    seed=11,
)
res = dfit.analyze_dfit(d.time_min, d.pressure_psi, d.rate_bpm)
print(res.summary())

## 2. Derive fracture design parameters

Pass the interpreted DFIT outputs directly into the design module.

In [ ]:
design = fracture_design_from_dfit(
    isip_psi=res.isip_psi,
    closure_pressure_psi=res.closure_pressure_psi,
    closure_time_min=res.closure_time_min or 80.0,
    t_pump_min=d.truth["t_pump_min"],
    fracture_height_ft=100.0,
    youngs_modulus_psi=4.0e6,
    design_half_length_ft=500.0,
    design_rate_bpm=10.0,
    tvd_ft=8500.0,
)
print(design_summary(design))

## 3. Sensitivity of treatment design to closure stress uncertainty

The single biggest driver of treatment pressure is closure stress.
Here we sweep closure stress across the plausible range from our
bootstrap analysis and show how the surface treating pressure changes.

In [ ]:
closure_range = np.linspace(res.closure_pressure_psi - 400,
                            res.closure_pressure_psi + 400, 30)
stp_range = []
for cp in closure_range:
    dp = fracture_design_from_dfit(
        res.isip_psi, cp, design.fracture_closure_time_min,
        d.truth["t_pump_min"], tvd_ft=8500.0)
    stp_range.append(dp.surface_treating_pressure_psi)

plt.figure(figsize=(7,4))
plt.plot(closure_range, stp_range, lw=2)
plt.axvline(res.closure_pressure_psi, color="r", ls="--",
            label=f"DFIT closure {res.closure_pressure_psi:.0f} psi")
plt.xlabel("Closure pressure (psi)")
plt.ylabel("Surface treating pressure (psi)")
plt.title("Treatment pressure sensitivity to closure stress")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Compare: design with vs without the DFIT

What does a naive design look like if the engineer just guesses?
A common default is to assume closure stress from the overburden gradient
(1.0 psi/ft) and guess 50% fluid efficiency.

In [ ]:
# Naive design (no DFIT)
tvd = 8500.0
naive_closure = 0.8 * tvd   # naive: assume 0.8 psi/ft gradient
naive_efficiency = 0.50
naive_CL = (1 - naive_efficiency) / (2 * np.sqrt(d.truth["t_pump_min"]))

# DFIT-calibrated design
dfit_closure = res.closure_pressure_psi
dfit_efficiency = design.fluid_efficiency

print("PARAMETER              NAIVE (no DFIT)    DFIT-CALIBRATED")
print("-" * 58)
print(f"Closure stress (psi)   {naive_closure:>15.0f}    {dfit_closure:>15.0f}")
print(f"Fluid efficiency (%)   {naive_efficiency*100:>15.0f}    {dfit_efficiency*100:>15.1f}")
print(f"C_L (ft/sqrt(min))     {naive_CL:>15.4f}    {design.leakoff_coefficient_ft_sqrtmin:>15.4f}")
print()
print("Implication: a naive design may over- or under-pump by a")
print("factor of 2-5x and exceed or fall short of the minimum")
print("treating pressure, wasting cost or failing to open the fracture.")

### The bottom line

The fracture design bridge shows exactly why DFITs are run before major
hydraulic fracture treatments:

- Without DFIT: engineer guesses closure stress, fluid efficiency, and C_L.
  Design errors of 2-5x on volume and 500-1000 psi on pressure are common.

- With DFIT: all four key inputs are measured. Treatment design is
  calibrated to the actual rock, not a gradient assumption.

The cost of a DFIT is one day of rig time. The cost of a mis-designed
main fracture treatment is the entire completion cost - often 10-20x more.